In [6]:
import os
import numpy as np
import pandas as pd
from pandas.io.parsers.readers import STR_NA_VALUES
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
custom_na_values = STR_NA_VALUES - {'None'}
df = pd.read_csv('../data/synthetic_coffee_health_10000.csv', keep_default_na=False, na_values=custom_na_values)

# Adjusting binary columns
df['Smoking'] = df['Smoking'].replace({0: 'No', 1: 'Yes'})
df['Alcohol_Consumption'] = df['Alcohol_Consumption'].replace({0: 'No', 1: 'Yes'})

# Fix for Sorting (Sets the exact order for plots)
sleep_order = ['Poor', 'Fair', 'Good', 'Excellent']
df['Sleep_Quality'] = pd.Categorical(df['Sleep_Quality'], categories=sleep_order, ordered=True)

# 2. Ordinal Encoding (Creating numerical versions for the heatmap)
df['Sleep_Quality_Num'] = df['Sleep_Quality'].map({'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}).astype(float)
df['Stress_Level_Num'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2}).astype(float)
df['Health_Issues_Num'] = df['Health_Issues'].map({'None': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3}).astype(float)

# Preview the first few rows
df.head()

,ID,Age,Gender,Country,Coffee_Intake,Caffeine_mg,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Health_Issues,Occupation,Smoking,Alcohol_Consumption,Sleep_Quality_Num,Stress_Level_Num,Health_Issues_Num
0,1,40,Male,Germany,3.5,328.1,7.5,Good,24.9,78,Low,14.5,None,Other,No,No,2.0,0.0,0.0
1,2,33,Male,Germany,1.0,94.1,6.2,Good,20.0,67,Low,11.0,None,Service,No,No,2.0,0.0,0.0
2,3,42,Male,Brazil,5.3,503.7,5.9,Fair,22.7,59,Medium,11.2,Mild,Office,No,No,1.0,1.0,1.0
3,4,53,Male,Germany,2.6,249.2,7.3,Good,24.7,71,Low,6.6,Mild,Other,No,No,2.0,0.0,1.0
4,5,32,Female,Spain,3.1,298.0,5.3,Fair,24.1,76,Medium,8.5,Mild,Student,No,Yes,1.0,1.0,1.0


In [9]:
# Cell 2: Feature Engineering

def engineer_features(data):
    df_eng = data.copy()
    
    # Age Buckets (Matched to teacher's bins)
    bins = [0, 25, 35, 45, 55, 65, 100]
    labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    df_eng['Age_Group'] = pd.cut(df_eng['Age'], bins=bins, labels=labels)
    
    # Coffee Strength
    df_eng['Caffeine_per_Cup'] = np.where(
        df_eng['Coffee_Intake'] > 0, 
        df_eng['Caffeine_mg'] / df_eng['Coffee_Intake'], 
        0
    )
    return df_eng

df_featured = engineer_features(df)

In [10]:
# Cell 3: Multi-Task Data Split

# Exclude ID, original text targets, and our new numeric targets from X
text_targets = ['Sleep_Quality', 'Stress_Level', 'Health_Issues']
num_targets = ['Sleep_Quality_Num', 'Stress_Level_Num', 'Health_Issues_Num']

X = df_featured.drop(columns=['ID'] + text_targets + num_targets)
y = df_featured[num_targets]

# Split 1: 70% Train, 30% Temp (Stratified on Sleep_Quality_Num)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y['Sleep_Quality_Num']
)

# Split 2: 15% Val, 15% Test (Stratified on Sleep_Quality_Num)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp['Sleep_Quality_Num']
)

print(f"\nStratified Split Complete:")
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


Stratified Split Complete:
Train: 7000 | Val: 1500 | Test: 1500


In [11]:
# Cell 4: Scikit-Learn Preprocessing Pipeline

numeric_features = [
    'Age', 'Coffee_Intake', 'Caffeine_mg', 'Sleep_Hours', 
    'BMI', 'Heart_Rate', 'Physical_Activity_Hours', 'Caffeine_per_Cup' 
]

categorical_features = [
    'Gender', 'Country', 'Occupation', 'Smoking', 'Alcohol_Consumption', 'Age_Group'
]

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit and Transform
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Get column names back for the CSVs
cat_cols = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
final_columns = numeric_features + list(cat_cols)

In [12]:
# --- 5. EXPORTING TO CSV ---
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save the preprocessor pipeline
joblib.dump(preprocessor, '../models/preprocessor.joblib')

# Combine X and Y back together to save as clean CSVs
train_df = pd.DataFrame(X_train_processed, columns=final_columns)
train_df[num_targets] = y_train.values

val_df = pd.DataFrame(X_val_processed, columns=final_columns)
val_df[num_targets] = y_val.values

test_df = pd.DataFrame(X_test_processed, columns=final_columns)
test_df[num_targets] = y_test.values

train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

# Optional: Save the full engineered dataframe before scaling
df_featured.to_csv('../data/processed/features_engineered.csv', index=False)

print("\n✓ Saved processed datasets to CSVs (train.csv, val.csv, test.csv)")


✓ Saved processed datasets to CSVs (train.csv, val.csv, test.csv)
